In [ ]:
import os
os.environ["HF_HOME"] = r"C:\memotion\cache\hf"
os.environ["TRANSFORMERS_CACHE"] = r"C:\memotion\cache\hf\transformers"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  

In [2]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN")

In [4]:
import sys, os, torch, transformers
print("PYTHON:", sys.executable)
print("TORCH :", torch.__version__)
print("TFM   :", transformers.__version__)
print("CUDA  :", torch.cuda.is_available())
print("HF_HOME:", os.environ.get("HF_HOME"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

PYTHON: C:\Users\ASUS\.conda\envs\sarcasm_gpu\python.exe
TORCH : 2.5.1+cu121
TFM   : 5.2.0
CUDA  : True
HF_HOME: C:\memotion\cache\hf
TRANSFORMERS_CACHE: C:\memotion\cache\hf\transformers


In [5]:
import transformers, huggingface_hub
print(transformers.__version__)
print(huggingface_hub.__version__)
from transformers import AutoTokenizer, AutoModel
print("✅ transformers import OK")

5.2.0
1.5.0
✅ transformers import OK


In [ ]:

import os, re, glob, json, math, random, shutil, time
from pathlib import Path


os.environ.setdefault('HF_HOME', r'C:\\memotion\\cache\\hf')
os.environ.setdefault('TRANSFORMERS_CACHE', r'C:\\memotion\\cache\\hf\\transformers')
HF_TOKEN = os.environ.get('HF_TOKEN')
print('HF_TOKEN set:', HF_TOKEN is not None)


import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from PIL import Image


from transformers import AutoTokenizer, AutoModel, CLIPVisionModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)


HF_TOKEN set: True
✅ device: cuda


In [ ]:

DATA_ROOT = r"C:\memotion\raw\memotion_dataset_7k"   


OUTPUT_ROOT = r"C:\memotion\runs_sarcasm"             

import os, glob

def _pick_first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


label_candidates = []
for pat in ["labels*.csv", "labels*.xlsx", "labels*.xls", "**/labels*.csv", "**/labels*.xlsx", "**/labels*.xls"]:
    label_candidates += glob.glob(os.path.join(DATA_ROOT, pat), recursive=True)


label_candidates_sorted = sorted(label_candidates, key=lambda p: (0 if p.lower().endswith(".csv") else 1, len(p)))
CSV_PATH = label_candidates_sorted[0] if label_candidates_sorted else None


img_exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
best_dir, best_count = None, 0
for root, dirs, files in os.walk(DATA_ROOT):
    c = sum(1 for f in files if f.lower().endswith(img_exts))
    if c > best_count:
        best_count = c
        best_dir = root

IMAGES_DIR = best_dir

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("DATA_ROOT   :", DATA_ROOT)
print("CSV_PATH    :", CSV_PATH)
print("IMAGES_DIR  :", IMAGES_DIR, f"(images found: {best_count})")
print("OUTPUT_ROOT :", OUTPUT_ROOT)

if CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find labels file under DATA_ROOT. Expected something like labels.csv / labels.xlsx. "
        f"DATA_ROOT={DATA_ROOT}"
    )
if IMAGES_DIR is None or best_count == 0:
    raise FileNotFoundError(
        "Could not find any images under DATA_ROOT. Please verify the dataset contains image files "
        "(.jpg/.png) and update DATA_ROOT accordingly."
    )


DATA_ROOT   : C:\memotion\raw\memotion_dataset_7k
CSV_PATH    : C:\memotion\raw\memotion_dataset_7k\labels.csv
IMAGES_DIR  : C:\memotion\raw\memotion_dataset_7k\images (images found: 6991)
OUTPUT_ROOT : C:\memotion\runs_sarcasm


In [ ]:

OUTPUT_ROOT = OUTPUT_ROOT
os.makedirs(OUTPUT_ROOT, exist_ok=True)


LR_LIST = [3e-5]
WD_LIST = [0.05]
DROPOUT_LIST = [0.5]
BATCH_LIST = [24]  
EPOCHS_LIST = [4]  
SELECT_BEST_BY = 'macro_f1'


KEEP_ONLY_BEST = True
SAVE_FP16_WEIGHTS = True
KEEP_PREDICTIONS_FOR_BEST = True
COMPRESS_PREDICTIONS_GZ = True

print('✅ Tuning space:', len(LR_LIST)*len(WD_LIST)*len(DROPOUT_LIST)*len(BATCH_LIST)*len(EPOCHS_LIST), 'configs')


✅ Tuning space: 1 configs


In [ ]:
if 'CSV_PATH' in globals() and CSV_PATH:
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"CSV_PATH not found: {CSV_PATH}")
    csv_path = CSV_PATH
else:
    csv_path = None

def find_first(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            hits_sorted = sorted(hits, key=lambda x: (("test" in x.lower()) + ("sample" in x.lower()), len(x)))
            return hits_sorted[0]
    return None

if csv_path is None:
    csv_path = find_first([
        "/kaggle/input/**/labels.csv",
        "/kaggle/input/**/train*.csv",
        "/kaggle/input/**/Train*.csv",
        "/kaggle/input/**/*memotion*/*.csv",
        "/kaggle/input/**/*.csv",
    ])
if csv_path is None:
    raise FileNotFoundError("Could not auto-find Memotion CSV under /kaggle/input. Attach dataset and retry.")

print("✅ Using CSV:", csv_path)
memotion_df = pd.read_csv(csv_path)
print("memotion_df shape:", memotion_df.shape)
print("Columns:", memotion_df.columns.tolist())


if 'IMAGES_DIR' in globals() and IMAGES_DIR:
    if not os.path.isdir(IMAGES_DIR):
        raise FileNotFoundError(f"IMAGES_DIR not found: {IMAGES_DIR}")
    images_dir = IMAGES_DIR
else:
    img_dirs = []
    for pat in ["/kaggle/input/**/images", "/kaggle/input/**/Images", "/kaggle/input/**/img", "/kaggle/input/**/Imgs", "/kaggle/input/**/memes"]:
        img_dirs.extend(glob.glob(pat, recursive=True))
    img_dirs = [d for d in img_dirs if os.path.isdir(d)]
    if not img_dirs:
        raise FileNotFoundError("Could not find an images directory under /kaggle/input. Check dataset mount.")
    images_dir = sorted(img_dirs, key=len)[0]
print("✅ Using images_dir:", images_dir)


if "image_name" not in memotion_df.columns:
    img_cands = [c for c in memotion_df.columns if re.search(r"(image|file|meme)", c, re.I)]
    if not img_cands:
        raise KeyError("No image filename column found.")
    memotion_df["image_name"] = memotion_df[img_cands[0]].astype(str)


if "binary_label" not in memotion_df.columns:
    use_col = "sarcasm" if "sarcasm" in memotion_df.columns else None
    if use_col is None:
        sarc_cands = [c for c in memotion_df.columns if re.search(r"sarc", c, re.I)]
        use_col = sarc_cands[0] if sarc_cands else None
    if use_col is None:
        raise KeyError("Could not infer sarcasm label column.")
    s = memotion_df[use_col].astype(str).str.strip().str.lower()
    is_num = s.str.fullmatch(r"-?\d+(\.\d+)?").fillna(False)
    binary = np.zeros(len(s), dtype=int)
    if is_num.any():
        num_vals = pd.to_numeric(s[is_num], errors="coerce").fillna(0).values
        binary[is_num.values] = (num_vals != 0).astype(int)
    neg_tokens = {"0","none","not_sarcastic","not sarcastic","no","na","nan","null","non-sarcastic","notsarcastic","not-sarcastic"}
    mask_str = (~is_num.values)
    if mask_str.any():
        binary[mask_str] = (~s[mask_str].isin(neg_tokens)).astype(int)
    memotion_df["binary_label"] = binary
    print(f"✅ Created binary_label from '{use_col}' with robust mapping.")
else:
    memotion_df["binary_label"] = pd.to_numeric(memotion_df["binary_label"], errors="coerce").fillna(0).astype(int)

print("binary_label distribution:\n", memotion_df["binary_label"].value_counts())

TEXT_COL = "text_corrected" if "text_corrected" in memotion_df.columns else ("text_ocr" if "text_ocr" in memotion_df.columns else None)
if TEXT_COL is None:
    raise KeyError("No usable text column found (text_corrected/text_ocr).")
print("✅ TEXT_COL:", TEXT_COL)

sample_paths = [os.path.join(images_dir, fn) for fn in memotion_df["image_name"].astype(str).head(50)]
exist_ct = sum([os.path.exists(p) for p in sample_paths])
print(f"✅ Sanity: {exist_ct}/50 sample images found.")


✅ Using CSV: C:\memotion\raw\memotion_dataset_7k\labels.csv
memotion_df shape: (6992, 9)
Columns: ['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour', 'sarcasm', 'offensive', 'motivational', 'overall_sentiment']
✅ Using images_dir: C:\memotion\raw\memotion_dataset_7k\images
✅ Created binary_label from 'sarcasm' with robust mapping.
binary_label distribution:
 binary_label
1    5448
0    1544
Name: count, dtype: int64
✅ TEXT_COL: text_corrected
✅ Sanity: 50/50 sample images found.


In [ ]:

train_df, temp_df = train_test_split(memotion_df, test_size=0.30, random_state=SEED, stratify=memotion_df["binary_label"])
val_df, test_df  = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["binary_label"])
print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)


Train/Val/Test: (4894, 10) (1049, 10) (1049, 10)


In [ ]:

EMOJI_RE = re.compile("[" 
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
"]+", flags=re.UNICODE)

def extract_emojis(text: str):
    if not isinstance(text, str): 
        return []
    return EMOJI_RE.findall(text)

emoji_counter = {}
for t in tqdm(train_df[TEXT_COL].astype(str).tolist(), desc="Building emoji vocab"):
    for e in extract_emojis(t):
        emoji_counter[e] = emoji_counter.get(e, 0) + 1

MAX_EMOJIS = 500
emoji_list = sorted(emoji_counter.items(), key=lambda x: x[1], reverse=True)[:MAX_EMOJIS]
emoji2id = {e:i+1 for i,(e,_) in enumerate(emoji_list)}  # 0=PAD
print("✅ emoji vocab size:", len(emoji2id))

def emojis_to_ids(text: str, max_k=16):
    ems = extract_emojis(text)
    ids = [emoji2id.get(e, 0) for e in ems][:max_k]
    if len(ids) < max_k:
        ids += [0]*(max_k-len(ids))
    return ids


Building emoji vocab:   0%|          | 0/4894 [00:00<?, ?it/s]

✅ emoji vocab size: 3


In [ ]:


import os, re, json, math
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from transformers import AutoTokenizer


TEXT_MODEL = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL,
    token=HF_TOKEN,
    use_fast=False,
)


resnet_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

clip_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                         std=[0.26862954,0.26130258,0.27577711]),
])

class MemeDataset(Dataset):
    def __init__(self, df, images_dir, text_col, max_len=64, max_emojis=16):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.text_col = text_col
        self.max_len = max_len
        self.max_emojis = max_emojis

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, str(r["image_name"]))
        y = int(r["binary_label"])
        text = str(r[self.text_col])

        ok = 1
        try:
            img = Image.open(img_path).convert("RGB")
            pixel_resnet = resnet_transform(img)
            pixel_clip   = clip_transform(img)
        except Exception:
            pixel_resnet = torch.zeros(3,224,224)
            pixel_clip   = torch.zeros(3,224,224)
            ok = 0

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        eids = torch.tensor(emojis_to_ids(text, self.max_emojis), dtype=torch.long)

        return {
            "pixel_resnet": pixel_resnet,
            "pixel_clip": pixel_clip,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "emoji_ids": eids,
            "label": torch.tensor(y, dtype=torch.float32),
            "ok": torch.tensor(ok, dtype=torch.long)
        }

def collate_fn(batch):
    batch = [b for b in batch if int(b["ok"]) == 1]
    if len(batch) == 0:
        return None
    out = {}
    for k in ["pixel_resnet","pixel_clip","input_ids","attention_mask","emoji_ids","label"]:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out

MAX_LEN = 64

train_ds = MemeDataset(train_df, images_dir, TEXT_COL, max_len=MAX_LEN)
val_ds   = MemeDataset(val_df, images_dir, TEXT_COL, max_len=MAX_LEN)
test_ds  = MemeDataset(test_df, images_dir, TEXT_COL, max_len=MAX_LEN)

print("✅ datasets ready")


y_tr = train_df["binary_label"].astype(int).values
class_counts = np.bincount(y_tr, minlength=2)
class_w = 1.0 / np.maximum(class_counts, 1)
sample_w = class_w[y_tr].astype(np.float32)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_w),
    num_samples=len(sample_w),
    replacement=True
)

print("Class counts:", class_counts, " | sampler ON")

✅ datasets ready
Class counts: [1081 3813]  | sampler ON


In [ ]:


import torch, torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel, CLIPVisionModel

class SarcasmPaperModel(nn.Module):
    def __init__(self, emoji_vocab_size, dropout=0.3):
        super().__init__()

        TEXT_MODEL = "ai4bharat/indic-bert"
        try:
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL, token=HF_TOKEN, use_safetensors=True)
        except Exception:
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL, token=HF_TOKEN)

        self.resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.resnet.fc = nn.Identity()        # 2048
        self.img_a_proj = nn.Linear(2048, 512)

        try:
            self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
        except Exception:
            self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        self.img_b_proj = nn.Linear(self.clip_vision.config.hidden_size, 512)

        self.img_fuse = nn.Linear(1024, 768)

        self.emoji_emb = nn.Embedding(emoji_vocab_size + 1, 256, padding_idx=0)
        self.emoji_proj = nn.Linear(256, 768)

        self.attn_ti = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)
        self.attn_te = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        self.gate_ie = nn.Sequential(nn.Linear(768 + 768, 768), nn.Sigmoid())

        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(768)

       
        self.head_text  = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_image = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_emoji = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_multi = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))

       
        self.sent_text  = nn.Linear(768, 1)
        self.sent_image = nn.Linear(768, 1)
        self.sent_emoji = nn.Linear(768, 1)

    def masked_emoji_mean(self, emoji_ids):
        emb = self.emoji_emb(emoji_ids)                   
        mask = (emoji_ids != 0).float().unsqueeze(-1)      
        denom = mask.sum(dim=1).clamp(min=1.0)
        pooled = (emb * mask).sum(dim=1) / denom
        return pooled

    def forward(self, pixel_resnet, pixel_clip, input_ids, attention_mask, emoji_ids):
        tout = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        Htext = tout.last_hidden_state
        htext = Htext[:, 0, :]

        fa2048 = self.resnet(pixel_resnet)
        fa = self.img_a_proj(fa2048)

        vout = self.clip_vision(pixel_values=pixel_clip)
        vb = vout.pooler_output if getattr(vout, "pooler_output", None) is not None else vout.last_hidden_state[:,0,:]
        fb = self.img_b_proj(vb)

        fimg = self.img_fuse(torch.cat([fa, fb], dim=-1))

        e = self.masked_emoji_mean(emoji_ids)
        femoji = self.emoji_proj(e)

        img_tok = fimg.unsqueeze(1)
        attn_ti, _ = self.attn_ti(Htext, img_tok, img_tok)
        H_ti = self.ln(Htext + self.dropout(attn_ti))

        emo_tok = femoji.unsqueeze(1)
        attn_te, _ = self.attn_te(H_ti, emo_tok, emo_tok)
        H_att = self.ln(H_ti + self.dropout(attn_te))
        h_att_cls = H_att[:, 0, :]

        g = self.gate_ie(torch.cat([fimg, femoji], dim=-1))
        fimg_g = g * fimg

        fused = self.ln(h_att_cls + fimg_g + femoji)

       
        logit_text  = self.head_text(htext).squeeze(-1)
        logit_image = self.head_image(fimg_g).squeeze(-1)
        logit_emoji = self.head_emoji(femoji).squeeze(-1)
        logit_multi = self.head_multi(fused).squeeze(-1)

       
        st_logit = self.sent_text(htext).squeeze(-1)
        si_logit = self.sent_image(fimg).squeeze(-1)
        se_logit = self.sent_emoji(femoji).squeeze(-1)

      
        st = torch.sigmoid(st_logit)
        si = torch.sigmoid(si_logit)
        se = torch.sigmoid(se_logit)
        incong = (torch.abs(st - si) + torch.abs(st - se) + torch.abs(si - se)) / 3.0

        return logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit

model = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=0.3).to(device)
print("✅ model ready (LOGITS + separate pixels)")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight    

✅ model ready (LOGITS + separate pixels)


In [ ]:

def gwo_optimize(branch_probs, y_true, n_wolves=20, iters=60, seed=SEED):
    rng = np.random.default_rng(seed)
    pop = rng.random((n_wolves, 4))
    pop = pop / pop.sum(axis=1, keepdims=True)

    def fitness(w):
        p = (branch_probs * w.reshape(1,4)).sum(axis=1)
        yhat = (p >= 0.5).astype(int)
        return f1_score(y_true, yhat, average="macro")

    scores = np.array([fitness(w) for w in pop])
    for t in range(iters):
        idx = np.argsort(scores)[::-1]
        alpha, beta, delta = pop[idx[0]], pop[idx[1]], pop[idx[2]]
        a = 2 - 2*(t/(iters-1+1e-9))
        for i in range(n_wolves):
            w = pop[i]
            r = rng.uniform(-1, 1, size=4)
            D_alpha = np.abs(alpha - w)
            D_beta  = np.abs(beta  - w)
            D_delta = np.abs(delta - w)
            w_new = (alpha - a*D_alpha + beta - a*D_beta + delta - a*D_delta)/3.0 + (a*0.1)*r
            w_new = np.clip(w_new, 0, 1)
            if w_new.sum() == 0:
                w_new = rng.random(4)
            w_new = w_new / w_new.sum()
            pop[i] = w_new
        scores = np.array([fitness(w) for w in pop])
    best_idx = np.argmax(scores)
    return pop[best_idx], float(scores[best_idx])


In [ ]:


import os, json, numpy as np
from copy import deepcopy
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

PATIENCE   = 2
CLIP_NORM  = 1.0
NUM_WORKERS = 0

FREEZE_BACKBONES = True
PARTIAL_UNFREEZE = True


USE_SAMPLER = False
USE_POS_WEIGHT = False


W_AUX_SENT   = 0.20
W_AUX_INCONG = 0.10


FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0

def make_loaders(batch_size):
    tr = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn,
                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False)
    va = DataLoader(val_ds, batch_size=max(8, batch_size), shuffle=False, collate_fn=collate_fn,
                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False)
    te = DataLoader(test_ds, batch_size=max(8, batch_size), shuffle=False, collate_fn=collate_fn,
                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False)
    return tr, va, te

@torch.no_grad()
def eval_probs(model, loader):
    model.eval()
    ys, p_multi_all, br_all = [], [], []
    for batch in loader:
        if batch is None:
            continue
        batch = {k: v.to(device) for k, v in batch.items()}
        logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit = model(
            batch["pixel_resnet"], batch["pixel_clip"],
            batch["input_ids"], batch["attention_mask"], batch["emoji_ids"]
        )
        y = batch["label"].detach().float().cpu().numpy()
        ys.append(y)

        p_text  = torch.sigmoid(logit_text).detach().cpu().numpy()
        p_image = torch.sigmoid(logit_image).detach().cpu().numpy()
        p_emoji = torch.sigmoid(logit_emoji).detach().cpu().numpy()
        p_multi = torch.sigmoid(logit_multi).detach().cpu().numpy()

        p_multi_all.append(p_multi)
        br_all.append(np.stack([p_text, p_image, p_emoji, p_multi], axis=1))

    y_true = np.concatenate(ys) if ys else np.array([])
    p_multi = np.concatenate(p_multi_all) if p_multi_all else np.array([])
    br = np.concatenate(br_all) if br_all else np.zeros((0,4))
    return y_true, p_multi, br

def metrics_from_probs(y_true, p, thr=0.5):
    if len(y_true)==0:
        return {"acc":0.0,"macro_f1":0.0,"auc":0.0}
    yhat = (p >= thr).astype(int)
    return {
        "acc": float(accuracy_score(y_true, yhat)),
        "macro_f1": float(f1_score(y_true, yhat, average="macro")),
        "auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true))==2 else 0.0
    }

def tune_threshold_macro_f1(y_true, p, grid=None):
    if len(y_true)==0:
        return 0.5, 0.0
    if grid is None:
        grid = np.linspace(0.05, 0.95, 37)
    best_t, best_f = 0.5, -1.0
    for t in grid:
        f = f1_score(y_true, (p>=t).astype(int), average="macro")
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def save_best_state(model, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(out_dir, "best_model.pt"))


def focal_bce_with_logits(logits, targets, alpha=0.75, gamma=2.0):
    """
    logits: (B,)
    targets: (B,) float {0,1}
    """
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    pt = targets * p + (1 - targets) * (1 - p)
    w = alpha * targets + (1 - alpha) * (1 - targets)
    loss = w * (1 - pt).pow(gamma) * bce
    return loss.mean()


def gwo_optimize(br_probs, y_true, n_wolves=18, iters=60, seed=42, metric="auc"):
    rng = np.random.default_rng(seed)
    if len(y_true)==0:
        return np.array([0.25,0.25,0.25,0.25], float)

    def score(w):
        w = np.clip(w,1e-8,1.0); w = w/w.sum()
        p = (br_probs * w.reshape(1,4)).sum(axis=1)
        if metric=="auc":
            return roc_auc_score(y_true, p) if len(np.unique(y_true))==2 else 0.0
        t,_ = tune_threshold_macro_f1(y_true, p)
        return f1_score(y_true, (p>=t).astype(int), average="macro")

    wolves = rng.random((n_wolves,4)); wolves = wolves/wolves.sum(axis=1, keepdims=True)
    alpha, beta, delta = wolves[0].copy(), wolves[1].copy(), wolves[2].copy()
    alpha_s = beta_s = delta_s = -1e9

    for t in range(iters):
        scores = np.array([score(w) for w in wolves])
        idx = np.argsort(scores)[::-1]
        wolves, scores = wolves[idx], scores[idx]
        if scores[0] > alpha_s: alpha, alpha_s = wolves[0].copy(), scores[0]
        if scores[1] > beta_s:  beta,  beta_s  = wolves[1].copy(), scores[1]
        if scores[2] > delta_s: delta, delta_s = wolves[2].copy(), scores[2]

        a = 2 - 2*(t/(iters-1+1e-9))
        new = []
        for i in range(n_wolves):
            X = wolves[i].copy()
            def upd(Xp):
                r1, r2 = rng.random(4), rng.random(4)
                A = 2*a*r1 - a
                C = 2*r2
                D = np.abs(C*Xp - X)
                return Xp - A*D
            Xn = (upd(alpha)+upd(beta)+upd(delta))/3.0
            Xn = np.clip(Xn,1e-8,1.0); Xn = Xn/Xn.sum()
            new.append(Xn)
        wolves = np.vstack(new)

    w_best = np.clip(alpha,1e-8,1.0); w_best = w_best/w_best.sum()
    return w_best


BEST_CFG_PATH = r"C:\memotion\configs\BEST_CONFIG.json"
best_cfg = json.load(open(BEST_CFG_PATH, "r"))

LR      = float(best_cfg.get("lr", 3e-5))
WD      = float(best_cfg.get("weight_decay", 0.05))
DROPOUT = float(best_cfg.get("dropout", 0.5))
BS      = int(best_cfg.get("batch_size", 24))
EPOCHS  = int(best_cfg.get("epochs", 10))

print("✅ SINGLE BEST config:", (LR, WD, DROPOUT, BS, EPOCHS))


run_name = f"best_lr{LR:g}_wd{WD:g}_do{DROPOUT:g}_bs{BS}_ep{EPOCHS}_FOCAL_NOSAMPLER"
run_dir = os.path.join(OUTPUT_ROOT, run_name)
os.makedirs(run_dir, exist_ok=True)

train_loader, val_loader, test_loader = make_loaders(BS)

model = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=DROPOUT).to(device)

if FREEZE_BACKBONES:
    for p in model.text_encoder.parameters(): p.requires_grad=False
    for p in model.resnet.parameters(): p.requires_grad=False
    for p in model.clip_vision.parameters(): p.requires_grad=False
    print("✅ Backbones frozen (FAST mode).")

if PARTIAL_UNFREEZE:
    if hasattr(model.text_encoder,"encoder") and hasattr(model.text_encoder.encoder,"layer"):
        for layer in model.text_encoder.encoder.layer[-2:]:
            for p in layer.parameters(): p.requires_grad=True
    if hasattr(model.resnet,"layer4"):
        for p in model.resnet.layer4.parameters(): p.requires_grad=True
    if hasattr(model.clip_vision,"vision_model") and hasattr(model.clip_vision.vision_model,"encoder"):
        blocks = model.clip_vision.vision_model.encoder.layers
        for blk in blocks[-2:]:
            for p in blk.parameters(): p.requires_grad=True
    print("✅ Partial unfreeze enabled.")

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)

use_cuda_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

best_val = -1e9
best_epoch = -1
no_improve = 0
best_state = None

for ep in range(1, EPOCHS+1):
    model.train()
    losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {ep}/{EPOCHS}", leave=False)

    for batch in pbar:
        if batch is None:
            continue
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_cuda_amp):
            logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit = model(
                batch["pixel_resnet"], batch["pixel_clip"],
                batch["input_ids"], batch["attention_mask"], batch["emoji_ids"]
            )

        y = batch["label"].float()

       
        loss_main = focal_bce_with_logits(logit_multi.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)

        
        loss_aux_sent = (
            focal_bce_with_logits(st_logit.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA) +
            focal_bce_with_logits(si_logit.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA) +
            focal_bce_with_logits(se_logit.float(), y, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
        ) / 3.0

       
        loss_aux_incong = torch.nn.functional.binary_cross_entropy(incong.float().clamp(0,1), y)

        loss = loss_main + W_AUX_SENT*loss_aux_sent + W_AUX_INCONG*loss_aux_incong

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())
        if len(losses) % 20 == 0:
            pbar.set_postfix(loss=float(np.mean(losses[-20:])))

    
    yv, pv, brv = eval_probs(model, val_loader)

    m_multi = metrics_from_probs(yv, pv)
    w_best = gwo_optimize(brv, yv.astype(int), metric="auc")
    p_ens = (brv * w_best.reshape(1,4)).sum(axis=1)
    m_ens = metrics_from_probs(yv, p_ens)

    val_metric = m_ens["auc"]
    print(f"[VAL] ep={ep} loss={np.mean(losses):.4f} | multi_auc={m_multi['auc']:.4f} | ens_auc={m_ens['auc']:.4f} w={w_best}")

    if val_metric > best_val + 1e-6:
        best_val = float(val_metric)
        best_epoch = ep
        no_improve = 0
        best_state = deepcopy(model.state_dict())

        thr_multi, _ = tune_threshold_macro_f1(yv, pv)
        thr_ens, _ = tune_threshold_macro_f1(yv, p_ens)

        snap = {
            "run": run_name,
            "best_epoch": best_epoch,
            "val_multi": m_multi,
            "val_multi_thr": float(thr_multi),
            "val_ens": m_ens,
            "val_ens_thr": float(thr_ens),
            "gwo_weights": w_best.tolist(),
            "focal_alpha": float(FOCAL_ALPHA),
            "focal_gamma": float(FOCAL_GAMMA),
            "sampler": False
        }
        json.dump(snap, open(os.path.join(run_dir, "best_metrics.json"), "w"), indent=2)

        model.load_state_dict(best_state)
        save_best_state(model, run_dir)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

print("✅ Training done. Best epoch:", best_epoch, "Best val AUC:", best_val)
print("✅ Run dir:", run_dir)

✅ SINGLE BEST config: (3e-05, 0.05, 0.5, 24, 10)


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weigh

...

[VAL] ep=3 loss=0.1997 | multi_auc=0.4887 | ens_auc=0.5292 w=[5.48908263e-01 1.15642523e-01 3.35449193e-01 2.09654204e-08]


Epoch 4/10:  86%|████████████████████████████████████████████████        | 175/204 [01:32<00:14,  1.94it/s, loss=0.165]C:\Users\ASUS\.conda\envs\sarcasm_gpu\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                                                       

[VAL] ep=4 loss=0.1665 | multi_auc=0.4742 | ens_auc=0.5091 w=[7.08751306e-02 3.02658807e-08 9.29124790e-01 4.91188876e-08]


Epoch 5/10:   4%|██▋                                                                   | 8/204 [00:04<01:44,  1.87it/s]C:\Users\ASUS\.conda\envs\sarcasm_gpu\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
                                                                                                                       

[VAL] ep=5 loss=0.1368 | multi_auc=0.4702 | ens_auc=0.5091 w=[2.09707827e-02 1.18598285e-07 9.79029050e-01 4.85534640e-08]
Early stopping.
✅ Training done. Best epoch: 3 Best val AUC: 0.5291911112986959
✅ Run dir: C:\memotion\runs_sarcasm\best_lr3e-05_wd0.05_do0.5_bs24_ep10_FOCAL_NOSAMPLER


In [ ]:


import json
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_run_dir = run_dir
metrics_file = os.path.join(best_run_dir,"best_metrics.json")

snap = json.load(open(metrics_file))

w_best = np.array(snap["gwo_weights"],dtype=float)

print("Loaded GWO weights:",w_best)

yt, pt, brt = eval_probs(model,test_loader)


mt_multi = metrics_from_probs(yt,pt)

thr_multi = snap.get("val_multi_thr",0.5)
mt_multi_tuned = metrics_from_probs(yt,pt,thr=thr_multi)


p_ens = (brt * w_best.reshape(1,4)).sum(axis=1)

mt_ens = metrics_from_probs(yt,p_ens)

thr_ens = snap.get("val_ens_thr",0.5)
mt_ens_tuned = metrics_from_probs(yt,p_ens,thr=thr_ens)

results = {
"test_multi_auc":mt_multi["auc"],
"test_multi_f1":mt_multi["macro_f1"],
"test_multi_acc":mt_multi["acc"],

"test_ens_auc":mt_ens["auc"],
"test_ens_f1":mt_ens["macro_f1"],
"test_ens_acc":mt_ens["acc"],

"test_ens_f1_tuned":mt_ens_tuned["macro_f1"],
"test_ens_acc_tuned":mt_ens_tuned["acc"],

"ensemble_weights":w_best.tolist()
}

print("\nFINAL RESULTS")
print(json.dumps(results,indent=2))

with open(os.path.join(best_run_dir,"final_eval_fixed.json"),"w") as f:
    json.dump(results,f,indent=2)

Loaded GWO weights: [5.48908263e-01 1.15642523e-01 3.35449193e-01 2.09654204e-08]

FINAL RESULTS
{
  "test_multi_auc": 0.5349892568718975,
  "test_multi_f1": 0.46920003001402577,
  "test_multi_acc": 0.7492850333651097,
  "test_ens_auc": 0.48703680182897785,
  "test_ens_f1": 0.1866414490489641,
  "test_ens_acc": 0.223069590085796,
  "test_ens_f1_tuned": 0.43813604713444027,
  "test_ens_acc_tuned": 0.7797902764537655,
  "ensemble_weights": [
    0.5489082634119983,
    0.11564252300143311,
    0.3354491926211483,
    2.096542035455365e-08
  ]
}


In [ ]:


import os, json, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, precision_recall_curve, auc


best_pt = os.path.join(run_dir, "best_model.pt")
model.load_state_dict(torch.load(best_pt, map_location=device))
model.eval()

snap = json.load(open(os.path.join(run_dir,"best_metrics.json")))
w_best = np.array(snap["gwo_weights"], dtype=float)
thr_multi = float(snap.get("val_multi_thr", 0.5))
thr_ens = float(snap.get("val_ens_thr", 0.5))

yt, pt, brt = eval_probs(model, test_loader)


p_ens = (brt * w_best.reshape(1,4)).sum(axis=1)


m_multi = metrics_from_probs(yt, pt, thr=thr_multi)
m_ens   = metrics_from_probs(yt, p_ens, thr=thr_ens)

out = {
    "test_multi": m_multi,
    "test_ens": m_ens,
    "thr_multi": thr_multi,
    "thr_ens": thr_ens,
    "ensemble_weights": w_best.tolist(),
    "test_size": int(len(yt)),
    "test_pos_frac": float(np.mean(yt)) if len(yt) else None
}

with open(os.path.join(run_dir,"metrics.json"),"w") as f:
    json.dump(out,f,indent=2)

print("✅ Saved metrics.json")
print(json.dumps(out, indent=2))


pred_df = pd.DataFrame({
    "y_true": yt.astype(int),
    "p_multi": pt,
    "p_ens": p_ens,
    "yhat_multi": (pt >= thr_multi).astype(int),
    "yhat_ens": (p_ens >= thr_ens).astype(int)
})
pred_df.to_csv(os.path.join(run_dir,"predictions.csv"), index=False)
print("✅ Saved predictions.csv")


rep = classification_report(yt.astype(int), (p_ens >= thr_ens).astype(int), digits=4)
with open(os.path.join(run_dir,"classification_report.txt"),"w") as f:
    f.write(rep)
print("✅ Saved classification_report.txt")


cm = confusion_matrix(yt.astype(int), (p_ens >= thr_ens).astype(int))
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix (Ensemble)")
plt.xlabel("Pred")
plt.ylabel("True")
plt.xticks([0,1]); plt.yticks([0,1])
for i in range(2):
    for j in range(2):
        plt.text(j,i,str(cm[i,j]), ha="center", va="center")
plt.tight_layout()
plt.savefig(os.path.join(run_dir,"confusion_matrix.png"), dpi=200)
plt.close()
print("✅ Saved confusion_matrix.png")


fpr, tpr, _ = roc_curve(yt.astype(int), p_ens)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f"Ensemble AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve (Ensemble)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(run_dir,"roc.png"), dpi=200)
plt.close()
print("✅ Saved roc.png")


prec, rec, _ = precision_recall_curve(yt.astype(int), p_ens)
pr_auc = auc(rec, prec)
plt.figure()
plt.plot(rec, prec, label=f"Ensemble PR-AUC={pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curve (Ensemble)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(run_dir,"pr.png"), dpi=200)
plt.close()
print("✅ Saved pr.png")

print("✅ FINAL EVAL COMPLETE →", run_dir)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14664\3390743662.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_pt, map_location=device

✅ Saved metrics.json
{
  "test_multi": {
    "acc": 0.6425166825548141,
    "macro_f1": 0.5199822819008153,
    "auc": 0.5316551826331777
  },
  "test_ens": {
    "acc": 0.7797902764537655,
    "macro_f1": 0.43813604713444027,
    "auc": 0.4952449750738259
  },
  "thr_multi": 0.7,
  "thr_ens": 0.05,
  "ensemble_weights": [
    0.5489082634119983,
    0.11564252300143311,
    0.3354491926211483,
    2.096542035455365e-08
  ],
  "test_size": 1049,
  "test_pos_frac": 0.7797902822494507
}
✅ Saved predictions.csv
✅ Saved classification_report.txt


C:\Users\ASUS\.conda\envs\sarcasm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\.conda\envs\sarcasm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\.conda\envs\sarcasm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

✅ Saved confusion_matrix.png
✅ Saved roc.png
✅ Saved pr.png
✅ FINAL EVAL COMPLETE → C:\memotion\runs_sarcasm\best_lr3e-05_wd0.05_do0.5_bs24_ep10_FOCAL_NOSAMPLER


# -------------mustrad final pipline---------------

In [22]:
pip install opencv-python


Note: you may need to restart the kernel to use updated packages.


In [23]:
import os, glob, pathlib
import cv2

MUSTARD_ROOT = r"C:\memotion\raw\MUStARD-dataset-master\MUStARD-dataset-master"
UTT_DIR = os.path.join(MUSTARD_ROOT, "raw_videos", "mmsd_raw_data", "utterances_final")
OUT_DIR = os.path.join(MUSTARD_ROOT, "_keyframes_utt")
os.makedirs(OUT_DIR, exist_ok=True)

vids = glob.glob(os.path.join(UTT_DIR, "*.mp4"))
print("Utterance videos found:", len(vids))
print("Sample:", vids[:5])

fail = 0
for vp in vids:
    stem = pathlib.Path(vp).stem          # e.g., "1_60"
    op = os.path.join(OUT_DIR, f"{stem}.jpg")
    if os.path.exists(op):
        continue

    cap = cv2.VideoCapture(vp)
    if not cap.isOpened():
        fail += 1
        continue

    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n//2))
    ok, frame = cap.read()
    cap.release()

    if ok and frame is not None:
        cv2.imwrite(op, frame)
    else:
        fail += 1

print("✅ Keyframes saved:", OUT_DIR)
print("Failed:", fail)

Utterance videos found: 690
Sample: ['C:\\memotion\\raw\\MUStARD-dataset-master\\MUStARD-dataset-master\\raw_videos\\mmsd_raw_data\\utterances_final\\1_10004.mp4', 'C:\\memotion\\raw\\MUStARD-dataset-master\\MUStARD-dataset-master\\raw_videos\\mmsd_raw_data\\utterances_final\\1_10009.mp4', 'C:\\memotion\\raw\\MUStARD-dataset-master\\MUStARD-dataset-master\\raw_videos\\mmsd_raw_data\\utterances_final\\1_1001.mp4', 'C:\\memotion\\raw\\MUStARD-dataset-master\\MUStARD-dataset-master\\raw_videos\\mmsd_raw_data\\utterances_final\\1_1003.mp4', 'C:\\memotion\\raw\\MUStARD-dataset-master\\MUStARD-dataset-master\\raw_videos\\mmsd_raw_data\\utterances_final\\1_10190.mp4']
✅ Keyframes saved: C:\memotion\raw\MUStARD-dataset-master\MUStARD-dataset-master\_keyframes_utt
Failed: 8


In [ ]:
import os, json, pandas as pd

MUSTARD_ROOT = r"C:\memotion\raw\MUStARD-dataset-master\MUStARD-dataset-master"
ANN_PATH = os.path.join(MUSTARD_ROOT, "data", "sarcasm_data.json")
OUT_DIR = os.path.join(MUSTARD_ROOT, "_eval")
os.makedirs(OUT_DIR, exist_ok=True)

data = json.load(open(ANN_PATH, "r", encoding="utf-8"))

rows = []
for clip_id, rec in data.items():  
    utter = rec.get("utterance", "")
    ctx = rec.get("context", [])
    ctx_text = " ".join(ctx) if isinstance(ctx, list) else str(ctx)
    text = (ctx_text + " " + str(utter)).strip()
    y = int(bool(rec.get("sarcasm", False)))

    rows.append({
        "video_stem": str(clip_id),
        "text": text,
        "binary_label": y,
        "image_name": f"{clip_id}.jpg"
    })

df = pd.DataFrame(rows)
print(df.head(3))
print("Total:", len(df), "Pos frac:", df["binary_label"].mean())

OUT_CSV = os.path.join(OUT_DIR, "mustard_eval.csv")
df.to_csv(OUT_CSV, index=False)
print("✅ Saved:", OUT_CSV)

  video_stem                                               text  binary_label  \
0       1_60  I never would have identified the fingerprints...             1   
1       1_70  This is one of my favorite places to kick back...             1   
2       1_80  Here we go. Pad thai, no peanuts. But does it ...             0   

  image_name  
0   1_60.jpg  
1   1_70.jpg  
2   1_80.jpg  
Total: 690 Pos frac: 0.5
✅ Saved: C:\memotion\raw\MUStARD-dataset-master\MUStARD-dataset-master\_eval\mustard_eval.csv


In [ ]:
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, precision_recall_curve, auc, f1_score, roc_auc_score

MUSTARD_ROOT = r"C:\memotion\raw\MUStARD-dataset-master\MUStARD-dataset-master"
EVAL_CSV = os.path.join(MUSTARD_ROOT, "_eval", "mustard_eval.csv")
KEYFRAMES_DIR = os.path.join(MUSTARD_ROOT, "_keyframes_utt")
OUT_DIR = os.path.join(MUSTARD_ROOT, "_eval", "results_multi_utt")
os.makedirs(OUT_DIR, exist_ok=True)


BEST_RUN_DIR = r"C:\memotion\runs_sarcasm\best_lr3e-05_wd0.05_do0.5_bs24_ep10_FOCAL_NOSAMPLER"
best_pt = os.path.join(BEST_RUN_DIR, "best_model.pt")
snap_path = os.path.join(BEST_RUN_DIR, "best_metrics.json")

mustard_df = pd.read_csv(EVAL_CSV)

TEXT_COL = "text"
images_dir = KEYFRAMES_DIR

mustard_ds = MemeDataset(mustard_df, images_dir, TEXT_COL, max_len=MAX_LEN)
mustard_loader = DataLoader(mustard_ds, batch_size=32, shuffle=False, collate_fn=collate_fn,
                            num_workers=0, pin_memory=True, persistent_workers=False)


model.load_state_dict(torch.load(best_pt, map_location=device))
model.eval()

snap = json.load(open(snap_path, "r"))
thr_multi = float(snap.get("val_multi_thr", 0.5))

y_true, p_multi, br = eval_probs(model, mustard_loader)
yhat = (p_multi >= thr_multi).astype(int)

metrics = {
    "mustard_size": int(len(y_true)),
    "pos_frac": float(np.mean(y_true)) if len(y_true) else None,
    "thr_multi": thr_multi,
    "acc": float((yhat == y_true).mean()) if len(y_true) else 0.0,
    "macro_f1": float(f1_score(y_true, yhat, average="macro")) if len(y_true) else 0.0,
    "auc": float(roc_auc_score(y_true, p_multi)) if len(np.unique(y_true))==2 else 0.0
}

with open(os.path.join(OUT_DIR, "mustard_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame({"y_true": y_true.astype(int), "p_multi": p_multi, "yhat": yhat}).to_csv(
    os.path.join(OUT_DIR, "mustard_predictions.csv"), index=False
)

rep = classification_report(y_true.astype(int), yhat.astype(int), digits=4)
with open(os.path.join(OUT_DIR, "mustard_classification_report.txt"), "w") as f:
    f.write(rep)

cm = confusion_matrix(y_true.astype(int), yhat.astype(int))
plt.figure()
plt.imshow(cm)
plt.title("MUStARD Confusion Matrix (utterance keyframes, p_multi)")
plt.xlabel("Pred"); plt.ylabel("True")
plt.xticks([0,1]); plt.yticks([0,1])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "mustard_confusion_matrix.png"), dpi=200)
plt.close()

fpr, tpr, _ = roc_curve(y_true.astype(int), p_multi)
roc_auc_val = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC={roc_auc_val:.4f}")
plt.plot([0,1], [0,1], "--")
plt.title("MUStARD ROC (p_multi)")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "mustard_roc.png"), dpi=200)
plt.close()

prec, rec, _ = precision_recall_curve(y_true.astype(int), p_multi)
pr_auc_val = auc(rec, prec)
plt.figure()
plt.plot(rec, prec, label=f"PR-AUC={pr_auc_val:.4f}")
plt.title("MUStARD PR (p_multi)")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "mustard_pr.png"), dpi=200)
plt.close()

print("✅ MUStARD evaluation complete:", OUT_DIR)
print(json.dumps(metrics, indent=2))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14664\11130380.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_pt, map_location=device)

✅ MUStARD evaluation complete: C:\memotion\raw\MUStARD-dataset-master\MUStARD-dataset-master\_eval\results_multi_utt
{
  "mustard_size": 682,
  "pos_frac": 0.49560117721557617,
  "thr_multi": 0.7,
  "acc": 0.5263929618768328,
  "macro_f1": 0.5252814981628542,
  "auc": 0.5221807485895142
}


# Ablation / Comparisom Study..........................................

In [27]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:


import os, json, math, copy, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore")

ABL_DEVICE = torch.device("cpu")
print("Ablation device:", ABL_DEVICE)

required_globals = ["train_ds", "val_ds", "test_ds", "collate_fn", "OUTPUT_ROOT"]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(
        f"Missing required notebook variables: {missing_globals}. Run your main notebook cells first, then run this ablation cell."
    )

ABL_OUT = Path(OUTPUT_ROOT) / "ablation_study_full_updated"
ABL_OUT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)

def safe_auc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true)
        y_prob = np.asarray(y_prob)
        if len(np.unique(y_true)) < 2:
            return 0.0
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return 0.0

def tune_threshold(y_true, y_prob):
    best_thr, best_f1 = 0.50, -1.0
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    for thr in np.arange(0.05, 0.96, 0.05):
        pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, pred, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, float(best_f1)

def get_first_valid_batch(loader):
    for b in loader:
        if b is not None:
            return b
    raise RuntimeError("No valid batch found in loader.")

def build_ablation_loaders(batch_size=8):
    num_workers_local = globals().get("NUM_WORKERS", 0)
    tr = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=num_workers_local,
        pin_memory=False,
        persistent_workers=False if num_workers_local == 0 else True
    )
    va = DataLoader(
        val_ds,
        batch_size=max(8, batch_size),
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers_local,
        pin_memory=False,
        persistent_workers=False if num_workers_local == 0 else True
    )
    te = DataLoader(
        test_ds,
        batch_size=max(8, batch_size),
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers_local,
        pin_memory=False,
        persistent_workers=False if num_workers_local == 0 else True
    )
    return tr, va, te

def get_max_ids_from_loader(loader, max_batches=None):
    max_token_id = 0
    max_emoji_id = 0
    checked = 0
    for b in loader:
        if b is None:
            continue
        if "input_ids" in b:
            max_token_id = max(max_token_id, int(b["input_ids"].max().item()))
        if "emoji_ids" in b:
            max_emoji_id = max(max_emoji_id, int(b["emoji_ids"].max().item()))
        checked += 1
        if max_batches is not None and checked >= max_batches:
            break
    return max_token_id, max_emoji_id

def batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out

def evaluate_model(model, loader, thr=0.5):
    model.eval()
    ys, probs = [], []
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            batch = batch_to_device(batch, ABL_DEVICE)
            y = batch["label"].float().view(-1).cpu().numpy().astype(int)
            out = model(batch)
            logits = out["logits"].view(-1)
            p = torch.sigmoid(logits).cpu().numpy()
            ys.extend(y.tolist())
            probs.extend(p.tolist())
    ys = np.asarray(ys).astype(int)
    probs = np.asarray(probs).astype(float)
    pred = (probs >= thr).astype(int)
    return {
        "acc": float(accuracy_score(ys, pred)),
        "macro_f1": float(f1_score(ys, pred, average="macro", zero_division=0)),
        "auc": safe_auc(ys, probs),
        "y_true": ys,
        "y_prob": probs
    }

def train_one_model(model_name, model, train_loader, val_loader, test_loader, epochs=3, lr=1e-3):
    print("\n" + "="*80)
    print(f"RUNNING (FULL UPDATED): {model_name}")
    print("="*80)
    model_dir = ABL_OUT / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(ABL_DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    best_state = None
    best_val_f1 = -1.0
    best_thr = 0.50
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for batch in train_loader:
            if batch is None:
                continue
            batch = batch_to_device(batch, ABL_DEVICE)
            y = batch["label"].float().view(-1)
            optimizer.zero_grad()
            out = model(batch)
            logits = out["logits"].view(-1)
            if logits.shape != y.shape:
                raise RuntimeError(f"{model_name}: logits {logits.shape} vs labels {y.shape}")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        val_eval = evaluate_model(model, val_loader, thr=0.50)
        thr, tuned_f1 = tune_threshold(val_eval["y_true"], val_eval["y_prob"])
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)) if losses else 0.0,
            "val_acc": val_eval["acc"],
            "val_macro_f1_at_0.5": val_eval["macro_f1"],
            "val_auc": val_eval["auc"],
            "best_thr_now": thr,
            "best_val_macro_f1_now": tuned_f1
        }
        hist.append(row)
        print(f"Epoch {epoch}: loss={row['train_loss']:.4f}, val_f1={tuned_f1:.4f}, thr={thr:.2f}")
        if tuned_f1 > best_val_f1:
            best_val_f1 = tuned_f1
            best_thr = thr
            best_state = copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    test_eval = evaluate_model(model, test_loader, thr=best_thr)
    pd.DataFrame(hist).to_csv(model_dir / "history.csv", index=False)
    pd.DataFrame({
        "y_true": test_eval["y_true"],
        "y_prob": test_eval["y_prob"],
        "y_pred": (test_eval["y_prob"] >= best_thr).astype(int)
    }).to_csv(model_dir / "predictions.csv", index=False)
    summary = {
        "model": model_name,
        "status": "ok",
        "error": "",
        "params_trainable": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "best_val_macro_f1": float(best_val_f1),
        "best_thr": float(best_thr),
        "test_acc": float(test_eval["acc"]),
        "test_macro_f1": float(test_eval["macro_f1"]),
        "test_auc": float(test_eval["auc"])
    }
    with open(model_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    return summary

class TextBiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, pad_idx=0, dropout=0.3):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.embedding = nn.Embedding(self.vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = hidden_dim * 2
    def forward(self, input_ids, attention_mask=None):
        max_id = int(input_ids.max().item())
        min_id = int(input_ids.min().item())
        if min_id < 0 or max_id >= self.vocab_size:
            raise RuntimeError(
                f"TextBiLSTMEncoder input_ids out of range: min={min_id}, max={max_id}, vocab_size={self.vocab_size}"
            )
        x = self.embedding(input_ids)
        out, _ = self.lstm(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            out = out * mask
            denom = mask.sum(dim=1).clamp(min=1.0)
            pooled = out.sum(dim=1) / denom
        else:
            pooled = out.mean(dim=1)
        return self.dropout(pooled)

class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, num_filters=128, kernels=(3,4,5), pad_idx=0, dropout=0.3):
        super().__init__()
        self.vocab_size = int(vocab_size)
        self.embedding = nn.Embedding(self.vocab_size, emb_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, num_filters, k) for k in kernels])
        self.dropout = nn.Dropout(dropout)
        self.out_dim = num_filters * len(kernels)
    def forward(self, input_ids, attention_mask=None):
        max_id = int(input_ids.max().item())
        min_id = int(input_ids.min().item())
        if min_id < 0 or max_id >= self.vocab_size:
            raise RuntimeError(
                f"TextCNNEncoder input_ids out of range: min={min_id}, max={max_id}, vocab_size={self.vocab_size}"
            )
        x = self.embedding(input_ids).transpose(1, 2)
        feats = []
        for conv in self.convs:
            z = F.relu(conv(x))
            z = F.max_pool1d(z, kernel_size=z.size(2)).squeeze(2)
            feats.append(z)
        return self.dropout(torch.cat(feats, dim=1))

class ResNet50Encoder(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        backbone = models.resnet50(weights=None)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.out_dim = in_features
    def forward(self, pixel_resnet):
        z = self.backbone(pixel_resnet)
        return self.dropout(z)

class SmallClipLikeEncoder(nn.Module):
    def __init__(self, out_dim=512, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(128, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = out_dim
    def forward(self, pixel_clip):
        x = self.cnn(pixel_clip).flatten(1)
        x = self.fc(x)
        return self.dropout(x)

class DLTextOnlyBiLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.fc = nn.Linear(self.text.out_dim, 1)
    def forward(self, batch):
        z = self.text(batch["input_ids"], batch["attention_mask"])
        return {"logits": self.fc(z).squeeze(-1)}

class DLTextOnlyCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextCNNEncoder(vocab_size)
        self.fc = nn.Linear(self.text.out_dim, 1)
    def forward(self, batch):
        z = self.text(batch["input_ids"], batch["attention_mask"])
        return {"logits": self.fc(z).squeeze(-1)}

class DLImageOnlyResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        self.image = ResNet50Encoder()
        self.fc = nn.Linear(self.image.out_dim, 1)
    def forward(self, batch):
        z = self.image(batch["pixel_resnet"])
        return {"logits": self.fc(z).squeeze(-1)}

class DLTextImageConcat(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.image = ResNet50Encoder()
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + self.image.out_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zi = self.image(batch["pixel_resnet"])
        z = torch.cat([zt, zi], dim=1)
        return {"logits": self.fc(z).squeeze(-1)}

class DLTextImageClipFusion(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.resnet = ResNet50Encoder()
        self.clip_like = SmallClipLikeEncoder(out_dim=512)
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + self.resnet.out_dim + self.clip_like.out_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zr = self.resnet(batch["pixel_resnet"])
        zc = self.clip_like(batch["pixel_clip"])
        z = torch.cat([zt, zr, zc], dim=1)
        return {"logits": self.fc(z).squeeze(-1)}

class HybridConcatEmoji(nn.Module):
    def __init__(self, vocab_size, emoji_vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.image = ResNet50Encoder()
        self.emoji_vocab_size = int(emoji_vocab_size)
        self.emoji_emb = nn.Embedding(self.emoji_vocab_size, 32, padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + self.image.out_dim + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zi = self.image(batch["pixel_resnet"])
        emoji_ids = batch["emoji_ids"]
        emo_max = int(emoji_ids.max().item())
        emo_min = int(emoji_ids.min().item())
        if emo_min < 0 or emo_max >= self.emoji_vocab_size:
            raise RuntimeError(
                f"Emoji ids out of range: min={emo_min}, max={emo_max}, emoji_vocab_size={self.emoji_vocab_size}"
            )
        ze = self.emoji_emb(emoji_ids).mean(dim=1)
        z = torch.cat([zt, zi, ze], dim=1)
        return {"logits": self.fc(z).squeeze(-1)}

class HybridCrossAttnNoGWO(nn.Module):
    def __init__(self, vocab_size, emoji_vocab_size):
        super().__init__()
        self.text = TextBiLSTMEncoder(vocab_size)
        self.image = ResNet50Encoder()
        self.emoji_vocab_size = int(emoji_vocab_size)
        self.emoji_emb = nn.Embedding(self.emoji_vocab_size, 32, padding_idx=0)
        self.q_text = nn.Linear(self.text.out_dim, 128)
        self.k_img  = nn.Linear(self.image.out_dim, 128)
        self.v_img  = nn.Linear(self.image.out_dim, 128)
        self.q_emoji = nn.Linear(32, 128)
        self.fc = nn.Sequential(
            nn.Linear(self.text.out_dim + 128 + 128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, batch):
        zt = self.text(batch["input_ids"], batch["attention_mask"])
        zi = self.image(batch["pixel_resnet"])
        emoji_ids = batch["emoji_ids"]
        emo_max = int(emoji_ids.max().item())
        emo_min = int(emoji_ids.min().item())
        if emo_min < 0 or emo_max >= self.emoji_vocab_size:
            raise RuntimeError(
                f"Emoji ids out of range: min={emo_min}, max={emo_max}, emoji_vocab_size={self.emoji_vocab_size}"
            )
        ze = self.emoji_emb(emoji_ids).mean(dim=1)
        qt = self.q_text(zt)
        ki = self.k_img(zi)
        vi = self.v_img(zi)
        qe = self.q_emoji(ze)
        attn_ti = torch.sigmoid((qt * ki).sum(dim=1, keepdim=True) / math.sqrt(128))
        fused_ti = attn_ti * vi
        attn_te = torch.sigmoid((qt * qe).sum(dim=1, keepdim=True) / math.sqrt(128))
        fused_te = attn_te * qe
        z = torch.cat([zt, fused_ti, fused_te], dim=1)
        return {"logits": self.fc(z).squeeze(-1)}

train_loader, val_loader, test_loader = build_ablation_loaders(8)
first_batch = get_first_valid_batch(train_loader)
print("Batch keys found:", list(first_batch.keys()))
required_keys = ["pixel_resnet", "pixel_clip", "input_ids", "attention_mask", "emoji_ids", "label"]
missing_keys = [k for k in required_keys if k not in first_batch]
if missing_keys:
    raise RuntimeError(f"Missing batch keys: {missing_keys}")

train_tok_max, train_emo_max = get_max_ids_from_loader(train_loader, max_batches=None)
val_tok_max, val_emo_max     = get_max_ids_from_loader(val_loader,   max_batches=None)
test_tok_max, test_emo_max   = get_max_ids_from_loader(test_loader,  max_batches=None)

global_token_max = max(train_tok_max, val_tok_max, test_tok_max)
global_emoji_max = max(train_emo_max, val_emo_max, test_emo_max)

vocab_size = int(global_token_max + 1 + 32)
emoji_vocab_size = int(global_emoji_max + 1 + 32)

print("Global max token id:", global_token_max)
print("Global max emoji id:", global_emoji_max)
print("Using safe vocab_size:", vocab_size)
print("Using safe emoji_vocab_size:", emoji_vocab_size)

model_specs = [
    ("dl_text_only_bilstm",      DLTextOnlyBiLSTM(vocab_size),                       3, 1e-3),
    ("dl_text_only_textcnn",     DLTextOnlyCNN(vocab_size),                          3, 1e-3),
    ("dl_image_only_resnet50",   DLImageOnlyResNet50(),                              3, 1e-4),
    ("dl_text_image_concat",     DLTextImageConcat(vocab_size),                      3, 1e-4),
    ("dl_text_image_clipfusion", DLTextImageClipFusion(vocab_size),                  3, 1e-4),
    ("hybrid_concat_emoji",      HybridConcatEmoji(vocab_size, emoji_vocab_size),    3, 1e-4),
    ("hybrid_crossattn_no_gwo",  HybridCrossAttnNoGWO(vocab_size, emoji_vocab_size), 3, 1e-4),
]

rows = [{
    "model": "proposed_full_model",
    "status": "",
    "error": "",
    "params_trainable": np.nan,
    "best_val_macro_f1": np.nan,
    "best_thr": np.nan,
    "test_acc": np.nan,
    "test_macro_f1": np.nan,
    "test_auc": np.nan
}]

for model_name, model_obj, epochs, lr in model_specs:
    try:
        row = train_one_model(model_name, model_obj, train_loader, val_loader, test_loader, epochs=epochs, lr=lr)
    except Exception as e:
        print(f"[FATAL] {model_name} -> {type(e).__name__}: {e}")
        row = {
            "model": model_name,
            "status": "failed",
            "error": f"{type(e).__name__}: {e}",
            "params_trainable": 0.0,
            "best_val_macro_f1": 0.0,
            "best_thr": 0.5,
            "test_acc": 0.0,
            "test_macro_f1": 0.0,
            "test_auc": 0.0
        }
        model_dir = ABL_OUT / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        with open(model_dir / "summary.json", "w", encoding="utf-8") as f:
            json.dump(row, f, indent=2)
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_path = ABL_OUT / "ablation_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved:", summary_path)
display(summary_df)


Ablation device: cpu
Batch keys found: ['pixel_resnet', 'pixel_clip', 'input_ids', 'attention_mask', 'emoji_ids', 'label']
Global max token id: 199964
Global max emoji id: 3
Using safe vocab_size: 199997
Using safe emoji_vocab_size: 36

RUNNING (FULL UPDATED): dl_text_only_bilstm
Epoch 1: loss=0.5369, val_f1=0.4842, thr=0.80
Epoch 2: loss=0.5170, val_f1=0.4849, thr=0.80
Epoch 3: loss=0.4718, val_f1=0.5157, thr=0.70

RUNNING (FULL UPDATED): dl_text_only_textcnn
Epoch 1: loss=0.5851, val_f1=0.5247, thr=0.75
Epoch 2: loss=0.4796, val_f1=0.4822, thr=0.55
Epoch 3: loss=0.3933, val_f1=0.5174, thr=0.85

RUNNING (FULL UPDATED): dl_image_only_resnet50
Epoch 1: loss=0.5700, val_f1=0.4976, thr=0.85
Epoch 2: loss=0.5568, val_f1=0.5112, thr=0.70
Epoch 3: loss=0.5540, val_f1=0.5129, thr=0.75

RUNNING (FULL UPDATED): dl_text_image_concat
Epoch 1: loss=0.5436, val_f1=0.4627, thr=0.80
Epoch 2: loss=0.5360, val_f1=0.5033, thr=0.70
Epoch 3: loss=0.5374, val_f1=0.5030, thr=0.70

RUNNING (FULL UPDATED): dl

,model,status,error,params_trainable,best_val_macro_f1,best_thr,test_acc,test_macro_f1,test_auc
0,proposed_full_model,,,NaN,NaN,NaN,NaN,NaN,NaN
1,dl_text_only_bilstm,ok,,25864065.0,0.515681,0.70,0.591992,0.495378,0.522759
2,dl_text_only_textcnn,ok,,25796993.0,0.524725,0.75,0.679695,0.508583,0.526083
3,dl_image_only_resnet50,ok,,23510081.0,0.512907,0.75,0.607245,0.481237,0.476577
4,dl_text_image_concat,ok,,49962177.0,0.503341,0.70,0.696854,0.482639,0.497531
5,dl_text_image_clipfusion,ok,,50252545.0,0.514433,0.75,0.678742,0.488256,0.475256
6,hybrid_concat_emoji,ok,,49971521.0,0.528546,0.75,0.687321,0.487512,0.471843
7,hybrid_crossattn_no_gwo,ok,,50066241.0,0.514046,0.75,0.649190,0.486058,0.491628
